In [1]:
import torch
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification
import transformers.modeling_utils


transformers.modeling_utils.check_torch_load_is_safe = lambda: None
print("Loading saved tensor datasets...")
train_data = torch.load("ebm_nlp_train_tokenized.pt")
test_data = torch.load("ebm_nlp_test_tokenized.pt")


BATCH_SIZE = 8

print(f"Initializing DataLoaders (Batch Size: {BATCH_SIZE})...")
train_dataloader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)


print("\nDownloading and initializing BioBERT architecture...")

model = AutoModelForSequenceClassification.from_pretrained(
    "dmis-lab/biobert-base-cased-v1.2",
    num_labels=4
)


print("\n" + "="*40)
print("ARCHITECTURE VERIFICATION")
print("="*40)


print(model.classifier)


sample_batch = next(iter(train_dataloader))

with torch.no_grad():
    outputs = model(
        input_ids=sample_batch['input_ids'],
        attention_mask=sample_batch['attention_mask'],
        labels=sample_batch['labels']
    )

print(f"\nBatch input shape:  {sample_batch['input_ids'].shape}")
print(f"Model output shape: {outputs.logits.shape}") 
print(f"Initial loss:       {outputs.loss.item():.4f}")

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading saved tensor datasets...
Initializing DataLoaders (Batch Size: 8)...



You passed `num_labels=4` which is incompatible to the `id2label` map of length `2`.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 25745.42it/s]
BertForSequenceClassification LOAD REPORT from: dmis-lab/biobert-base-cased-v1.2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    


ARCHITECTURE VERIFICATION
Linear(in_features=768, out_features=4, bias=True)

Batch input shape:  torch.Size([8, 512])
Model output shape: torch.Size([8, 4])
Initial loss:       1.2125


In [2]:
import torch
from torch.optim import AdamW
import numpy as np


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")


model.to(device)


optimizer = AdamW(model.parameters(), lr=2e-5)


EPOCHS = 3 

def flat_accuracy(preds, labels):
    pred_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()
    return np.sum(pred_flat == labels_flat) / len(labels_flat)

print("\n" + "="*40)
print("STARTING TRAINING LOOP")
print("="*40)

for epoch in range(EPOCHS):
    print(f"\n======== Epoch {epoch + 1} / {EPOCHS} ========")
    
    print("Training...")
    model.train()
    total_train_loss = 0
    
    for step, batch in enumerate(train_dataloader):
        
        b_input_ids = batch['input_ids'].to(device)
        b_attn_mask = batch['attention_mask'].to(device)
        b_labels = batch['labels'].to(device)
        
        
        model.zero_grad()
        
        
        outputs = model(
            input_ids=b_input_ids,
            attention_mask=b_attn_mask,
            labels=b_labels
        )
        loss = outputs.loss
        total_train_loss += loss.item()
        loss.backward()
        optimizer.step()
        
        
        if step % 100 == 0 and step != 0:
            print(f"  Batch {step:>4} of {len(train_dataloader)}  |  Loss: {loss.item():.4f}")
            
    avg_train_loss = total_train_loss / len(train_dataloader)
    print(f"  -> Average Training Loss: {avg_train_loss:.4f}")
    
    
    print("\nRunning Validation on Test Set...")
    model.eval() 
    
    total_eval_accuracy = 0
    total_eval_loss = 0
    
    
    with torch.no_grad():
        for batch in test_dataloader:
            b_input_ids = batch['input_ids'].to(device)
            b_attn_mask = batch['attention_mask'].to(device)
            b_labels = batch['labels'].to(device)
            
            outputs = model(
                input_ids=b_input_ids,
                attention_mask=b_attn_mask,
                labels=b_labels
            )
            
            total_eval_loss += outputs.loss.item()
            
            logits = outputs.logits.detach().cpu().numpy()
            label_ids = b_labels.to('cpu').numpy()
            
            total_eval_accuracy += flat_accuracy(logits, label_ids)
            
    avg_val_accuracy = total_eval_accuracy / len(test_dataloader)
    avg_val_loss = total_eval_loss / len(test_dataloader)
    
    print(f"  -> Validation Loss: {avg_val_loss:.4f}")
    print(f"  -> Validation Accuracy: {avg_val_accuracy:.4f}")

print("\n" + "="*40)
print("TRAINING COMPLETE!")
print("="*40)

Training on device: cuda

STARTING TRAINING LOOP

======== Epoch 1 / 3 ========
Training...
  Batch  100 of 599  |  Loss: 1.0375
  Batch  200 of 599  |  Loss: 0.8544
  Batch  300 of 599  |  Loss: 0.8286
  Batch  400 of 599  |  Loss: 1.0378
  Batch  500 of 599  |  Loss: 0.2750
  -> Average Training Loss: 0.8082

Running Validation on Test Set...
  -> Validation Loss: 0.4604
  -> Validation Accuracy: 0.8198

======== Epoch 2 / 3 ========
Training...
  Batch  100 of 599  |  Loss: 0.6142
  Batch  200 of 599  |  Loss: 0.3111
  Batch  300 of 599  |  Loss: 0.3703
  Batch  400 of 599  |  Loss: 0.2779
  Batch  500 of 599  |  Loss: 0.2988
  -> Average Training Loss: 0.4974

Running Validation on Test Set...
  -> Validation Loss: 0.4316
  -> Validation Accuracy: 0.8594

======== Epoch 3 / 3 ========
Training...
  Batch  100 of 599  |  Loss: 0.1354
  Batch  200 of 599  |  Loss: 0.6766
  Batch  300 of 599  |  Loss: 0.5174
  Batch  400 of 599  |  Loss: 0.2652
  Batch  500 of 599  |  Loss: 0.2761
  -

In [3]:
import os
from transformers import AutoTokenizer



print("Loading official BioBERT tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("dmis-lab/biobert-base-cased-v1.2")

output_dir = './saved_biobert_pico_model/'


if not os.path.exists(output_dir):
    os.makedirs(output_dir)

print(f"Saving model and tokenizer to {output_dir}...")


id2label = {0: "Complete", 1: "Missing P", 2: "Missing I", 3: "Missing O"}
label2id = {"Complete": 0, "Missing P": 1, "Missing I": 2, "Missing O": 3}

model.config.id2label = id2label
model.config.label2id = label2id


model_to_save = model.module if hasattr(model, 'module') else model
model_to_save.save_pretrained(output_dir)


tokenizer.save_pretrained(output_dir)

print("Save complete.")

Loading official BioBERT tokenizer...
Saving model and tokenizer to ./saved_biobert_pico_model/...


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.78it/s]

Save complete.


In [5]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import classification_report

model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

all_preds = []
all_labels = []

print("Running evaluation on the test set...")


with torch.no_grad():
    for batch in test_dataloader:
        b_input_ids = batch['input_ids'].to(device)
        b_input_mask = batch['attention_mask'].to(device)
        b_labels = batch['labels'].to(device)

        outputs = model(b_input_ids, token_type_ids=None, attention_mask=b_input_mask)
        
        logits = outputs.logits
        preds = torch.argmax(logits, dim=1).flatten()
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(b_labels.cpu().numpy())


class_names = ["Complete", "Missing P", "Missing I", "Missing O"]



cm = confusion_matrix(all_labels, all_preds)
print(cm)
print(classification_report(all_labels, all_preds))


# plt.figure(figsize=(8, 6))
# sns.heatmap(
#     cm, 
#     annot=True,    
#     fmt='d',        
#     cmap='Blues',   
#     xticklabels=class_names, 
#     yticklabels=class_names
# )

# plt.title('BioBERT PICO Classification - Confusion Matrix', fontsize=14, pad=15)
# plt.ylabel('True Label', fontsize=12, fontweight='bold')
# plt.xlabel('Predicted Label', fontsize=12, fontweight='bold')
# plt.xticks(rotation=45)
# plt.yticks(rotation=0)

# # Save the plot to your Drive / local folder so you can use it in your report
# plt.tight_layout()
# plt.savefig('./saved_biobert_pico_model/confusion_matrix.png', dpi=300)
# plt.show()

Running evaluation on the test set...
[[90  1  0  4]
 [ 4 35  7  1]
 [ 1  2 19  2]
 [ 0  0  4 19]]
              precision    recall  f1-score   support

           0       0.95      0.95      0.95        95
           1       0.92      0.74      0.82        47
           2       0.63      0.79      0.70        24
           3       0.73      0.83      0.78        23

    accuracy                           0.86       189
   macro avg       0.81      0.83      0.81       189
weighted avg       0.87      0.86      0.86       189

